# Unit 5: Jaccard coefficient calculations

**Task (from the unit page):** the table shows pathological test results for three people. Calculate the
Jaccard distance (dissimilarity) for (Jack, Mary), (Jack, Jim) and (Jim, Mary).

| Name | Gender | Fever | Cough | Test-1 | Test-2 | Test-3 | Test-4 |
|---|---|---|---|---|---|---|---|
| Jack | M | Y | N | P | N | N | A |
| Mary | F | Y | N | P | A | P | N |
| Jim  | M | Y | P | N | N | N | A |

I worked it out by hand first, then checked it in code two ways: my own function and SciPy's.

## Step 1: decide which columns count

Jaccard is for **asymmetric binary** attributes. "Asymmetric" means a 1 matters more than a 0. Two
people both having a fever tells you something. Two people both *not* having a rare symptom tells you
almost nothing, because most people don't have it.

- **Gender** is symmetric. Male and female carry equal weight, so it is left out.
- **Fever, Cough and Test-1 to Test-4** are asymmetric. A positive result is the interesting event.

## Step 2: convert to 1s and 0s

Y (yes) and P (positive) become **1**. N (no) and A (absent) become **0**.

| Name | Fever | Cough | Test-1 | Test-2 | Test-3 | Test-4 |
|---|---|---|---|---|---|---|
| Jack | 1 | 0 | 1 | 0 | 0 | 0 |
| Mary | 1 | 0 | 1 | 0 | 1 | 0 |
| Jim  | 1 | 1 | 0 | 0 | 0 | 0 |

## Step 3: count the matches and mismatches for each pair

- f11 = both are 1
- f10 = first person 1, second person 0
- f01 = first person 0, second person 1
- f00 = both 0. **Jaccard ignores this one**, which is the whole point of it.

$$\text{Jaccard distance} = \frac{f_{01} + f_{10}}{f_{01} + f_{10} + f_{11}}$$

| Pair | f11 | f10 | f01 | Working | Distance |
|---|---|---|---|---|---|
| Jack, Mary | 2 (fever, test-1) | 0 | 1 (test-3) | 1 / 3 | **0.33** |
| Jack, Jim | 1 (fever) | 1 (test-1) | 1 (cough) | 2 / 3 | **0.67** |
| Jim, Mary | 1 (fever) | 1 (cough) | 2 (test-1, test-3) | 3 / 4 | **0.75** |

In [1]:
import numpy as np                                   # numpy: fast arrays for the 1/0 values
from scipy.spatial.distance import jaccard           # SciPy's own Jaccard distance, used as an independent check

# Map each letter in the table to a binary value: Y (yes) and P (positive) become 1, N (no) and A (absent) become 0
to_bin = {"Y": 1, "P": 1, "N": 0, "A": 0}  # letter -> 1 or 0
# Each person's six asymmetric columns, copied from the task table in order: Fever, Cough, Test-1 to Test-4
# (Gender is left out on purpose because it is a symmetric attribute)
raw = {  # each person's six asymmetric results
    "Jack": ["Y", "N", "P", "N", "N", "A"],          # Jack's results
    "Mary": ["Y", "N", "P", "A", "P", "N"],          # Mary's results
    "Jim":  ["Y", "P", "N", "N", "N", "A"],          # Jim's results
}
# Convert every letter to its 1/0 value and store each person as a numpy array
people = {name: np.array([to_bin[v] for v in vals]) for name, vals in raw.items()}  # convert letters to 1/0 arrays

def jaccard_distance(a, b):                          # my own version of the formula, so the working is visible
    f11 = np.sum((a == 1) & (b == 1))                # f11: columns where BOTH people have a 1
    f10 = np.sum((a == 1) & (b == 0))                # f10: first person 1, second person 0
    f01 = np.sum((a == 0) & (b == 1))                # f01: first person 0, second person 1
    return f11, f10, f01, (f01 + f10) / (f01 + f10 + f11)   # mismatches divided by every column with at least one 1

# The answers published on the module's "Hints and Solution" page, to check against
published = {("Jack", "Mary"): 0.33, ("Jack", "Jim"): 0.67, ("Jim", "Mary"): 0.75}  # answers from the module's Hints and Solution page
for (p, q), expected in published.items():           # go through each of the three pairs
    f11, f10, f01, mine = jaccard_distance(people[p], people[q])   # my calculation, with its counts
    sp = jaccard(people[p], people[q])               # SciPy's calculation of the same pair
    # print the counts and all three answers side by side so they can be compared by eye
    print(f"{p:>4} vs {q:<4}  f11={f11} f10={f10} f01={f01}  mine={mine:.2f}  scipy={sp:.2f}  published={expected}")  # counts and all three answers side by side
    assert round(mine, 2) == expected and round(sp, 2) == expected   # stop with an error if anything disagrees
print("All three match the published answers.")    # only reached if every assert passed

Jack vs Mary  f11=2 f10=0 f01=1  mine=0.33  scipy=0.33  published=0.33
Jack vs Jim   f11=1 f10=1 f01=1  mine=0.67  scipy=0.67  published=0.67
 Jim vs Mary  f11=1 f10=1 f01=2  mine=0.75  scipy=0.75  published=0.75
All three match the published answers.


## What happens if you get the column choice wrong?

The easy mistake is to keep Gender, or to treat every column as symmetric. Below I try both.

In [2]:
# Mistake 1: keep Gender as if it were just another binary column (M = 1, F = 0)
gender = {"Jack": 1, "Mary": 0, "Jim": 1}                                   # each person's gender as 1 or 0
with_gender = {n: np.append(v, gender[n]) for n, v in people.items()}      # add it as a seventh column

# Mistake 2: simple matching distance, which counts shared 0s ("both negative") as agreement too
def simple_matching_distance(a, b):  # the wrong measure, for comparison
    return np.mean(a != b)                       # share of ALL columns where the two people differ

for (p, q), expected in published.items():       # same three pairs as before
    g = jaccard(with_gender[p], with_gender[q])  # Jaccard distance with the Gender column wrongly included
    smd = simple_matching_distance(people[p], people[q])   # simple matching on the correct six columns
    # show the correct answer next to both mistakes
    print(f"{p:>4} vs {q:<4}  correct={expected}  with gender={g:.2f}  simple matching={smd:.2f}")  # correct answer next to both mistakes

Jack vs Mary  correct=0.33  with gender=0.50  simple matching=0.17
Jack vs Jim   correct=0.67  with gender=0.50  simple matching=0.33
 Jim vs Mary  correct=0.75  with gender=0.80  simple matching=0.50


## What I took from it

- **The ranking is the answer, not the number.** Jack and Mary are the most alike (0.33). Jim and Mary are
  the least alike (0.75). In a clinical setting that is what you would act on, for example grouping
  patients with similar results.
- **Simple matching makes everyone look more alike.** It counts "both tested negative" as agreement. With six
  columns and lots of negatives, that pulls every distance down. Jaccard exists to stop that.
- **Adding Gender changes the ranking**, not just the numbers. Jack goes from clearly closest to Mary (0.33
  vs 0.67) to a dead tie between Mary and Jim (0.50 each). Jack and Jim share a gender, so one column that
  says nothing about the illness wipes out the difference in their symptoms.
- That is a small version of a bigger point from this module. What you feed a similarity measure decides
  what "similar" means. On real patient data, a careless column choice could group people by a protected
  characteristic instead of by their symptoms. That is an ethical problem, not just a maths one.